# Guia de plotagem: pandas `melt` e matplotlib

Este notebook **não é uma análise**. É um guia: cada célula explica **um pedaço** do código usado nas Aulas 5 e 6.

O exemplo o tempo todo são as **Tabelas 5** (participação de homens e mulheres entre as pessoas desocupadas, 2012 T1 e 2026 T1).

**Mapa**

1. Bibliotecas
2. Abrir o CSV
3. `rename` e `merge`
4. Formato largo × formato longo e o `melt`
5. Anatomia do matplotlib (`Figure` e `Axes`)
6. Barras (`bar`, `barh`, agrupadas)
7. Linha de referência, títulos, cores, legenda
8. Dois gráficos lado a lado
9. Dispersão (`scatter`) e rótulos (`annotate`)
10. Linhas (`plot`)
11. Seaborn (`barplot` com `hue`)
12. Receita final

In [ ]:
# No terminal, uma vez: python -m pip install pandas matplotlib seaborn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")

## 1. O que cada biblioteca faz

| Nome no código | Pacote | Papel |
|----------------|--------|--------|
| `pd` | pandas | tabela (DataFrame): abrir, juntar, derreter, filtrar |
| `plt` | matplotlib.pyplot | desenhar o gráfico (figura, eixos, barras, linhas) |
| `sns` | seaborn | atalho em cima do matplotlib (tema e `barplot` com `hue`) |

`sns.set_theme(style="whitegrid", palette="Set2")`

- `style="whitegrid"`: fundo branco com grades — mais fácil de ler valores.
- `palette="Set2"`: conjunto de cores pastel, bom para categorias.

In [ ]:
path_2012 = r"C:\Users\janai\OneDrive\Documentos\python-class\dados\Tabela5-sem_emprego_2012.csv"
path_2026 = r"C:\Users\janai\OneDrive\Documentos\python-class\dados\Tabela5-sem_emprego_2026.csv"

## 2. Abrir o CSV — cada argumento do `read_csv`

```python
pd.read_csv(caminho, sep=";", decimal=",", encoding="utf-8-sig")
```

| Argumento | O que é | Por que nas Tabelas 5 |
|-----------|---------|------------------------|
| `sep=";"` | caractere que separa colunas | o arquivo usa ponto e vírgula, não vírgula |
| `decimal=","` | símbolo das casas decimais | no CSV está `54,7`; o pandas precisa virar `54.7` |
| `encoding="utf-8-sig"` | como as letras estão gravadas | UTF-8 com BOM: lê `Código`, `Amapá` sem estragar o acento |

Se o encoding estiver errado, nomes de estado não batem no `merge` depois.

In [ ]:
t2012 = pd.read_csv(path_2012, sep=";", decimal=",", encoding="utf-8-sig")
t2026 = pd.read_csv(path_2026, sep=";", decimal=",", encoding="utf-8-sig")
t2012.head()

## 3. `rename` — nomes curtos para o gráfico

`rename(columns={nome_antigo: nome_novo})` troca o rótulo da coluna. Não mexe nos números.

Sem isso, você passaria a vida escrevendo `Desocupados - mulheres (2012 T1)`.

In [ ]:
t2012 = t2012.rename(columns={
    "Desocupados - homens (2012 T1)": "homens_2012",
    "Desocupados - mulheres (2012 T1)": "mulheres_2012",
})
t2026 = t2026.rename(columns={
    "Desocupados - homens (2026 T1)": "homens_2026",
    "Desocupados - mulheres (2026 T1)": "mulheres_2026",
})
t2012.columns.tolist()

## 4. `merge` — juntar as duas tabelas

```python
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
```

| Peça | Significado |
|------|-------------|
| `t2012.merge(t2026, ...)` | a tabela da esquerda recebe colunas da direita |
| `on=[...]` | chaves: só junta a linha se esses campos forem iguais nos dois lados |
| `how="inner"` | fica só o que existe **nos dois** arquivos (aqui, as 27 UFs) |

Outros `how`: `left` (fica tudo da esquerda), `outer` (união de tudo).

In [ ]:
comp = t2012.merge(t2026, on=["Sigla", "Código", "Estado"], how="inner")
comp["var_mulheres_pp"] = (comp["mulheres_2026"] - comp["mulheres_2012"]).round(1)
print("linhas:", len(comp), "| colunas:", list(comp.columns))
comp.head()

`var_mulheres_pp` é uma coluna **nova**, calculada:

- `comp["mulheres_2026"] - comp["mulheres_2012"]` — pandas subtrai coluna a coluna, linha a linha
- `.round(1)` — uma casa decimal (pontos percentuais)

## 5. Formato largo × formato longo

O matplotlib (e o seaborn com `hue`) quase sempre quer **formato longo**:

- **Largo (wide):** cada ano é uma coluna. Uma linha = um estado.
- **Longo (long / tidy):** cada linha = um estado **em um ano**. Ano vira valor, não nome de coluna.

Olhe só 3 estados, largo:

In [ ]:
mini = comp.loc[comp["Estado"].isin(["Rondônia", "Acre", "Bahia"]),
                ["Estado", "mulheres_2012", "mulheres_2026"]]
mini

### `melt` — cada argumento

```python
mini.melt(id_vars="Estado", value_vars=["mulheres_2012", "mulheres_2026"],
          var_name="Ano", value_name="Participacao")
```

| Argumento | Função |
|-----------|--------|
| `id_vars` | colunas que **permanecem** (identificam a linha: Estado, Sigla…) |
| `value_vars` | colunas que vão **derreter**: cada uma vira várias linhas |
| `var_name` | nome da coluna nova que guarda o **nome antigo** da coluna derretida |
| `value_name` | nome da coluna nova que guarda o **número** |

3 estados × 2 anos = **6 linhas**.

In [ ]:
mini_longo = mini.melt(
    id_vars="Estado",
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao",
)
mini_longo

Os nomes `mulheres_2012` ainda são feios no gráfico. `map` troca valor por valor na coluna:

In [ ]:
mini_longo["Ano"] = mini_longo["Ano"].map({
    "mulheres_2012": "2012 T1",
    "mulheres_2026": "2026 T1",
})
mini_longo

In [ ]:
longo = comp.melt(
    id_vars=["Estado", "Sigla"],
    value_vars=["mulheres_2012", "mulheres_2026"],
    var_name="Ano",
    value_name="Participacao_mulheres",
)
longo["Ano"] = longo["Ano"].map({
    "mulheres_2012": "2012 T1",
    "mulheres_2026": "2026 T1",
})
print("largo:", comp.shape, "→ longo:", longo.shape)
longo.head(6)

## 6. Anatomia do matplotlib

Dois objetos:

```
Figure  (a folha de papel inteira)
 └── Axes  (um gráfico nessa folha — pode haver 2, 3…)
      ├── barras, linhas, pontos
      ├── título, eixos X e Y
      └── legenda
```

```python
fig, ax = plt.subplots(figsize=(8, 4))
```

| Peça | Significado |
|------|-------------|
| `plt.subplots()` | cria a folha **e** o gráfico de uma vez |
| `fig` | a Figure (folha) — use para `tight_layout`, `suptitle`, `savefig` |
| `ax` | o Axes (o gráfico) — é nele que você desenha |
| `figsize=(largura, altura)` | tamanho em **polegadas** |

Prefira `ax.bar(...)` a `plt.bar(...)`. Com dois gráficos, `plt.bar` não sabe em qual Axes pintar.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.set_title("Folha vazia: só o Axes, ainda sem dados")
ax.set_xlabel("eixo X")
ax.set_ylabel("eixo Y")
fig.tight_layout()
plt.show()

`fig.tight_layout()` aperta as margens para título e eixos não cortarem.
`plt.show()` manda o notebook desenhar. Sem ele, em alguns ambientes o gráfico não aparece.

## 7. `bar` — barras verticais

```python
ax.bar(x, altura)
```

- 1º argumento: posição / rótulo no eixo X (aqui, o estado)
- 2º argumento: altura da barra (o número)

Começamos com 5 estados para não poluir.

In [ ]:
cinco = comp.sort_values("mulheres_2026", ascending=False).head(5)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(cinco["Estado"], cinco["mulheres_2026"])
ax.set_title("Participação das mulheres entre desocupados — 2026 T1")
ax.set_xlabel("Estado")
ax.set_ylabel("% mulheres")
fig.tight_layout()
plt.show()

## 8. `barh` — barras horizontais

`barh(y, largura)`: o estado vai para o **eixo Y**. Melhor quando os nomes são longos (27 UFs).

`sort_values("mulheres_2026")` ordena **antes** de plotar: a maior barra fica em cima se o eixo Y cresce para cima.

In [ ]:
cinco_h = cinco.sort_values("mulheres_2026")

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cinco_h["Estado"], cinco_h["mulheres_2026"])
ax.set_xlabel("% mulheres entre desocupados (2026 T1)")
ax.set_ylabel("")
ax.set_title("barh: nomes no eixo Y")
fig.tight_layout()
plt.show()

## 9. Barras agrupadas — dois anos no mesmo gráfico

Uma barra por estado não basta: queremos **2012 e 2026** lado a lado.

Ideia: o eixo X é posição `0, 1, 2…`. Cada grupo ganha duas barras deslocadas.

```python
x = range(n)                    # 0, 1, 2, 3, 4
ax.bar([i - 0.2 for i in x], valores_2012, width=0.4, label="2012 T1")
ax.bar([i + 0.2 for i in x], valores_2026, width=0.4, label="2026 T1")
```

| Peça | Significado |
|------|-------------|
| `width=0.4` | largura de cada barra (em unidades do eixo) |
| `- 0.2` e `+ 0.2` | empurra 2012 para a esquerda e 2026 para a direita |
| `label=` | texto que vai aparecer na **legenda** |
| `ax.set_xticks` / `set_xticklabels` | troca 0,1,2… pelos nomes dos estados |

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = range(len(cinco))
ax.bar([i - 0.2 for i in x], cinco["mulheres_2012"], width=0.4, label="2012 T1")
ax.bar([i + 0.2 for i in x], cinco["mulheres_2026"], width=0.4, label="2026 T1")
ax.set_xticks(list(x))
ax.set_xticklabels(cinco["Estado"], rotation=20, ha="right")
ax.set_ylabel("% mulheres entre desocupados")
ax.set_title("Barras agrupadas: dois anos")
ax.legend()
fig.tight_layout()
plt.show()

`rotation=20` gira o texto do eixo. `ha="right"` alinha o fim da palavra no tick (não fica torto).

Outra forma (Aula 6): `align="edge"` com `width` positivo e negativo — uma barra cresce para um lado do tick, a outra para o outro.

## 10. `axhline` e `axvline` — linha de referência

| Função | Desenha |
|--------|---------|
| `ax.axhline(50)` | linha **horizontal** em y = 50 (metade das desocupações) |
| `ax.axvline(0)` | linha **vertical** em x = 0 (útil em gráfico de variação) |

| Argumento | Função |
|-----------|--------|
| `color` | cor (`"gray"`, `"black"`, `"#66c2a5"`) |
| `linestyle="--"` | tracejado (`"-"` sólido, `":"` pontilhado) |
| `linewidth` | espessura |

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(cinco_h["Estado"], cinco_h["mulheres_2026"])
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("% mulheres")
ax.set_title("axvline(50): acima da linha, mulheres são maioria")
fig.tight_layout()
plt.show()

## 11. Cores estado a estado

`color=` pode ser **uma** cor ou uma **lista** do mesmo tamanho das barras.

A lista por compreensão abaixo: verde se mulheres ≥ 50%, azul se não.

In [ ]:
todos = comp.sort_values("mulheres_2026")
cores = ["#66c2a5" if v >= 50 else "#8da0cb" for v in todos["mulheres_2026"]]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(todos["Estado"], todos["mulheres_2026"], color=cores)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("% mulheres entre desocupados (2026 T1)")
ax.set_title("Uma cor por regra: verde = maioria feminina")
fig.tight_layout()
plt.show()

## 12. Dois Axes na mesma Figure

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharex=True)
```

| Peça | Significado |
|------|-------------|
| `1, 2` | 1 linha, 2 colunas → `axes[0]` à esquerda, `axes[1]` à direita |
| `sharex=True` | os dois gráficos compartilham a escala do X (comparação justa) |
| `fig.suptitle` | título da **folha** inteira, acima dos dois |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8), sharex=True)

for ax, col, titulo in [
    (axes[0], "mulheres_2012", "2012 T1"),
    (axes[1], "mulheres_2026", "2026 T1"),
]:
    dados = comp.sort_values(col)
    cores = ["#66c2a5" if v >= 50 else "#8da0cb" for v in dados[col]]
    ax.barh(dados["Estado"], dados[col], color=cores)
    ax.axvline(50, color="gray", linestyle="--", linewidth=1)
    ax.set_title(titulo)
    ax.set_xlabel("% mulheres")

fig.suptitle("Os dois anos lado a lado (sharex: mesma escala)")
fig.tight_layout()
plt.show()

## 13. Gráfico de variação (valores negativos)

`barh` aceita número negativo: a barra cresce para a **esquerda**.
`axvline(0)` marca o zero. Laranja = alta da participação feminina; verde = queda.

In [ ]:
var = comp.sort_values("var_mulheres_pp")
cores = ["#fc8d62" if v > 0 else "#66c2a5" for v in var["var_mulheres_pp"]]

fig, ax = plt.subplots(figsize=(9, 8))
ax.barh(var["Estado"], var["var_mulheres_pp"], color=cores)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Variação da participação feminina (pp)")
ax.set_title("2012 → 2026: quem subiu e quem desceu")
fig.tight_layout()
plt.show()

## 14. `scatter` e `annotate`

`scatter(x, y)`: cada estado vira um **ponto** (não uma barra).

```python
ax.scatter(comp["mulheres_2012"], comp["mulheres_2026"], s=70)
ax.annotate(texto, (x, y), textcoords="offset points", xytext=(4, 4))
```

| Peça | Significado |
|------|-------------|
| `s=70` | tamanho do ponto (área) |
| `annotate(texto, (x, y))` | escreve o texto junto ao ponto |
| `textcoords="offset points"` | o deslocamento é em **pixels**, não em unidades do eixo |
| `xytext=(4, 4)` | 4 pontos para a direita e 4 para cima — o rótulo não cobre o ponto |

A diagonal `y = x` (pontilhada): acima dela, a participação feminina **subiu** de 2012 para 2026.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(comp["mulheres_2012"], comp["mulheres_2026"], s=70)
for _, r in comp.iterrows():
    ax.annotate(
        r["Sigla"],
        (r["mulheres_2012"], r["mulheres_2026"]),
        textcoords="offset points",
        xytext=(4, 4),
        fontsize=8,
    )
lims = [44, 68]
ax.plot(lims, lims, linestyle=":", color="gray")
ax.set_xlabel("% mulheres 2012 T1")
ax.set_ylabel("% mulheres 2026 T1")
ax.set_title("Dispersão: cada ponto é um estado")
fig.tight_layout()
plt.show()

`comp.iterrows()` devolve `(índice, linha)` para cada UF. Usamos `_` no índice porque não precisamos dele.

`ax.plot(lims, lims)` aqui **não** é série temporal: são dois pontos iguais em X e Y, ligados, formando a diagonal.

## 15. `plot` — gráfico de linhas

```python
ax.plot(xs, ys, marker="o", markersize=6, linewidth=1.5, label=sigla)
```

| Peça | Significado |
|------|-------------|
| `xs, ys` | coordenadas ligadas na ordem em que aparecem |
| `marker="o"` | círculo em cada ponto (`"s"` quadrado, `"x"` cruz) |
| `markersize` | tamanho do marcador |
| `linewidth` | espessura da linha |
| `label` | nome na legenda |

Com só dois anos, a “evolução” de cada estado é um segmento. O `groupby("Sigla")` da Aula 5 faz o mesmo quando há muitos anos: um `plot` por grupo.

In [ ]:
destaques = ["RO", "AC", "BA", "SP", "TO"]

fig, ax = plt.subplots(figsize=(8, 4))
for sigla, grupo in comp[comp["Sigla"].isin(destaques)].groupby("Sigla"):
    ax.plot(
        [2012, 2026],
        [grupo["mulheres_2012"].iloc[0], grupo["mulheres_2026"].iloc[0]],
        marker="o",
        markersize=6,
        linewidth=1.5,
        label=sigla,
    )

ax.set_xticks([2012, 2026])
ax.set_xlabel("Ano")
ax.set_ylabel("% mulheres entre desocupados")
ax.set_title("plot + groupby: uma linha por estado")
ax.legend(title="UF", ncol=1, loc="upper left", bbox_to_anchor=(1.02, 1))
fig.tight_layout()
plt.show()

### Legenda fora do gráfico

```python
ax.legend(title="UF", ncol=1, loc="upper left", bbox_to_anchor=(1.02, 1))
```

| Peça | Significado |
|------|-------------|
| `title` | cabeçalho da caixinha |
| `ncol` | quantas colunas de itens (com 27 UFs, `ncol=3` cabe melhor) |
| `loc="upper left"` | que **canto da caixinha** fica âncora |
| `bbox_to_anchor=(1.02, 1)` | onde essa âncora senta: um pouco à **direita** do Axes (`1.0` é a borda) |

`iloc[0]` pega o primeiro (e único) valor daquele estado depois do `groupby`.

## 16. Seaborn `barplot` com `hue` — o `melt` entra em cena

O seaborn lê o DataFrame **longo**:

```python
sns.barplot(data=longo, y="Estado", x="Participacao_mulheres", hue="Ano", order=ordem, ax=ax)
```

| Peça | Significado |
|------|-------------|
| `data=` | o DataFrame |
| `y=` / `x=` | colunas: categoria e número |
| `hue="Ano"` | cor (e agrupamento) segundo o ano — por isso o melt precisou criar a coluna `Ano` |
| `order=` | ordem dos estados no eixo; senão o seaborn usa ordem alfabética |
| `ax=ax` | desenha **neste** Axes matplotlib (não cria outra figura) |

Sem `melt`, não existe coluna `Ano` para o `hue`.

In [ ]:
ordem = comp.sort_values("mulheres_2026")["Estado"]

fig, ax = plt.subplots(figsize=(10, 10))
sns.barplot(
    data=longo,
    y="Estado",
    x="Participacao_mulheres",
    hue="Ano",
    order=ordem,
    ax=ax,
)
ax.axvline(50, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Participação das mulheres entre as pessoas desocupadas (%)")
ax.set_ylabel("")
ax.set_title("seaborn.barplot: hue = ano (dados longos)")
ax.legend(title="")
fig.tight_layout()
plt.show()

## 17. Outros pedaços de pandas usados nas aulas

| Código | O que faz |
|--------|-----------|
| `df.dropna(axis=1, how="all")` | joga fora colunas 100% vazias (Tabela 4) |
| `str(c).isdigit()` | a coluna é um ano? (`"1991"` → True) |
| `astype(int)` / `pd.to_numeric` | texto vira número |
| `sort_values("col")` | ordena linhas |
| `df.loc[df["Estado"] == "Rondônia"]` | filtra linhas |
| `.iloc[0]` | primeira linha do recorte, como Series |
| `groupby("Sigla")` | um bloco por UF |
| `rank(ascending=False)` | 1º, 2º, 3º… |
| `.corr()` | correlação de Pearson entre duas colunas |
| `isin([...])` | a UF está nesta lista? |

In [ ]:
print("Média simples das 27 UFs (não ponderada pela população)")
print("  mulheres 2012:", round(comp["mulheres_2012"].mean(), 1))
print("  mulheres 2026:", round(comp["mulheres_2026"].mean(), 1))
print()
print("Correlação 2012 × 2026 (participação feminina):",
      round(comp["mulheres_2012"].corr(comp["mulheres_2026"]), 2))
print()
print("Ranking 2026 (1 = maior participação feminina)")
rank = comp.assign(pos=comp["mulheres_2026"].rank(ascending=False).astype(int))
print(rank.sort_values("pos")[["pos", "Estado", "mulheres_2026"]].head(5).to_string(index=False))

## 18. Receita: do CSV ao gráfico

1. **Abrir** com os argumentos certos (`sep`, `decimal`, `encoding`).
2. **Renomear** colunas longas.
3. **Juntar** tabelas (`merge`) se houver mais de um ano/arquivo.
4. Decidir o tipo de gráfico:
   - categorias lado a lado com `hue` → **`melt`** + `sns.barplot` (ou barras agrupadas na mão)
   - um número por estado → `barh`
   - dois números por estado → `scatter`
   - evolução no tempo → `plot` (um `groupby` por série)
5. Criar `fig, ax = plt.subplots(...)`.
6. Desenhar no `ax`.
7. Título, eixos, linha de referência, legenda.
8. `tight_layout()` e `show()`.

Abaixo, o caminho curto reutilizado: melt + seaborn.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(
    data=mini_longo,
    x="Estado",
    y="Participacao",
    hue="Ano",
    ax=ax,
)
ax.axhline(50, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("% mulheres entre desocupados")
ax.set_title("Receita mínima: 3 estados, melt + barplot")
fig.tight_layout()
plt.show()

### Mini-glossário matplotlib

| Quer… | Use |
|-------|-----|
| criar a folha e o gráfico | `fig, ax = plt.subplots(figsize=(…))` |
| dois gráficos | `fig, axes = plt.subplots(1, 2)` |
| barras verticais | `ax.bar` |
| barras horizontais | `ax.barh` |
| linha | `ax.plot` |
| pontos | `ax.scatter` |
| escrever no ponto | `ax.annotate` |
| linha y = constante | `ax.axhline` |
| linha x = constante | `ax.axvline` |
| título / eixos | `set_title`, `set_xlabel`, `set_ylabel` |
| limites do eixo | `set_ylim`, `set_xlim` |
| ticks | `set_xticks`, `set_xticklabels` |
| legenda | `ax.legend` |
| título da folha | `fig.suptitle` |
| não cortar texto | `fig.tight_layout()` |
| aparecer no notebook | `plt.show()` |